# Batalla Naval: Aprendizaje por Refuerzo (MLP Único de 4 Neuronas)

Notebook listo para **Google Colab** (GPU/CPU) con observación matricial `(100, 4)` para un MLP de **4 neuronas de entrada** con pesos compartidos por casilla, motor vectorizado por bloques de 20 rondas, entrenamiento REINFORCE de doble escala, bots de referencia (**Bot Aleatorio** y **Bot Heurístico Hunt & Target**) y persistencia en Google Drive.

In [ ]:
# 1. Configurar entorno en Colab (clonar si hace falta y verificar GPU/CPU)
import os, sys
from pathlib import Path

if not Path("battleship_rl").exists():
    !git clone https://github.com/sergioicc/batal-war-MLP.git
    %cd batal-war-MLP

!pip install -q -r requirements.txt

import torch
print("PyTorch:", torch.__version__, "| CUDA disponible:", torch.cuda.is_available())

In [ ]:
# (Opcional en Colab) Montar Google Drive para guardar checkpoints y métricas
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
# 2. Ejecutar suite de pruebas de integridad (motor, matriz (100, 4), baselines y doble gradiente)
!pytest battleship_rl/tests -v

In [ ]:
# 3. Configuración y entrenamiento del primer bot MLP
from battleship_rl.config import Config
from battleship_rl.learning.trainer import DualScaleReinforceTrainer
from battleship_rl.utils.checkpoint import resolve_checkpoint_dir
from battleship_rl.utils.metrics import compare_agents

cfg = Config(
    num_blocks=100,        # 100 bloques x 20 rondas = 2000 partidas
    eval_every_blocks=10,
    eval_num_games=100,
    save_every_blocks=25,
)

ckpt_dir = resolve_checkpoint_dir(cfg.checkpoint_dir)
trainer = DualScaleReinforceTrainer(cfg)

before = compare_agents(trainer.agent, num_games=cfg.eval_num_games, eval_seed=cfg.eval_seed_base)
print("Antes de entrenar ->", before)

history = trainer.train(num_blocks=cfg.num_blocks, checkpoint_dir=ckpt_dir, verbose=True)

after = compare_agents(trainer.agent, num_games=cfg.eval_num_games, eval_seed=cfg.eval_seed_base)
print("Después de entrenar ->", after)

In [ ]:
# 4. Visualizar curvas de entrenamiento y comparación contra baselines
import pandas as pd
import matplotlib.pyplot as plt

train_df = pd.DataFrame(history["training_history"])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_df["block"], train_df["mean_shots"], label="MLP (media por bloque de 20)", linewidth=2)
ax.fill_between(
    train_df["block"],
    train_df["mean_shots"] - train_df["std_shots"],
    train_df["mean_shots"] + train_df["std_shots"],
    alpha=0.2,
    label="± 1 std",
)
ax.axhline(after["random"]["mean"], color="red", linestyle="--", label=f"Bot Aleatorio ({after['random']['mean']:.1f})")
ax.axhline(after["heuristic"]["mean"], color="green", linestyle="-.", label=f"Heurístico Hunt & Target ({after['heuristic']['mean']:.1f})")
ax.set_xlabel("Bloque (20 rondas)")
ax.set_ylabel("Disparos para ganar")
ax.set_title("Evolución del agente MLP vs Baselines")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()